# Λ-Type Three-Level: Coherent Population Trapping (CPT)

When two coherent fields simultaneously drive a three-level Λ system, the
atoms can be optically pumped into a **dark state** — a coherent
superposition of the two ground states that is completely decoupled from
the excited state.

## Level structure

```
         |1⟩ ──────────────── (excited)
        /    \
   Ω_p /      \ Ω_c
      /        \     γ₁₀ = γ₁₂ = 1.0
    |0⟩        |2⟩  (ground states)
```

## Physics

The Hamiltonian in the rotating frame has a zero-eigenvalue eigenstate —
the **dark state** — when the two-photon resonance condition holds
($\delta_p = \delta_c$):

$$
|D\rangle = \frac{\Omega_c\,|0\rangle - \Omega_p\,|2\rangle}
                  {\sqrt{\Omega_p^2 + \Omega_c^2}}
$$

Atoms pumped into $|D\rangle$ stop scattering photons entirely. The probe
absorption spectrum then shows a **narrow transparency window** centred on
$\delta_p = \delta_c$: the CPT (or EIT) dip.

The width of the dip is set by the ground-state decoherence rate $\gamma_g$,
which can be much narrower than the excited-state linewidth $\gamma_{10}$.
This makes CPT-based dark resonances extremely sharp — the basis for
CPT atomic clocks and all-optical magnetometry.

In [ ]:
import numpy as np
from maxwellbloch import mb_solve, plot

## Without coupling: Doppler-broadened absorption

First we solve for the probe alone (coupling Rabi frequency = 0, i.e.
no coupling field). The probe sees a Doppler-broadened Lorentzian — the
baseline against which the CPT dip will be compared.

A thermal velocity distribution is included
(`thermal_std = 10 γ`) to reproduce realistic lineshape broadening.

In [ ]:
mb_solve_json_no_coupling = """
{
  "atom": {
    "num_states": 3,
    "decays": [
      {"channels": [[0, 1]], "rate": 1.0},
      {"channels": [[1, 2]], "rate": 1.0}
    ],
    "fields": [
      {
        "label": "probe",
        "coupled_levels": [[0, 1]],
        "detuning": 0.0,
        "detuning_positive": true,
        "rabi_freq": 1.0e-3,
        "rabi_freq_t_func": "gaussian",
        "rabi_freq_t_args": {"ampl": 1.0, "centre": 0.0, "fwhm": 1.5}
      },
      {
        "label": "coupling",
        "coupled_levels": [[1, 2]],
        "detuning": 0.0,
        "detuning_positive": false,
        "rabi_freq": 0.0,
        "rabi_freq_t_func": "ramp_onoff",
        "rabi_freq_t_args": {"ampl": 1.0, "fwhm": 0.2, "on": -2.0, "off": 8.0}
      }
    ]
  },
  "t_min": -2.0,
  "t_max": 8.0,
  "t_steps": 200,
  "z_min": 0.0,
  "z_max": 1.0,
  "z_steps": 10,
  "z_steps_inner": 2,
  "interaction_strengths": [2.0, 0.0],
  "velocity_classes": {
    "thermal_width": 10.0,
    "thermal_delta_min": -30.0,
    "thermal_delta_max":  30.0,
    "thermal_delta_steps": 12,
    "thermal_delta_inner_min": -3.0,
    "thermal_delta_inner_max":  3.0,
    "thermal_delta_inner_steps": 6
  },
  "savefile": "mbs-lambda-cpt-no-coupling"
}
"""

mbs_no_c = mb_solve.MBSolve().from_json_str(mb_solve_json_no_coupling)
mbs_no_c.mbsolve(recalc=False)
print("Done")

## With coupling: CPT dark resonance

Now we add a strong CW coupling field ($\Omega_c = 5\gamma$) that is
switched on before the probe pulse arrives. The two-photon resonance
condition ($\delta_p = \delta_c = 0$) is satisfied on resonance, so a
sharp transparency window opens at $\delta_p = 0$.

In [ ]:
mb_solve_json_coupling = """
{
  "atom": {
    "num_states": 3,
    "decays": [
      {"channels": [[0, 1]], "rate": 1.0},
      {"channels": [[1, 2]], "rate": 1.0}
    ],
    "fields": [
      {
        "label": "probe",
        "coupled_levels": [[0, 1]],
        "detuning": 0.0,
        "detuning_positive": true,
        "rabi_freq": 1.0e-3,
        "rabi_freq_t_func": "gaussian",
        "rabi_freq_t_args": {"ampl": 1.0, "centre": 0.0, "fwhm": 1.5}
      },
      {
        "label": "coupling",
        "coupled_levels": [[1, 2]],
        "detuning": 0.0,
        "detuning_positive": false,
        "rabi_freq": 5.0,
        "rabi_freq_t_func": "ramp_onoff",
        "rabi_freq_t_args": {"ampl": 1.0, "fwhm": 0.2, "on": -2.0, "off": 8.0}
      }
    ]
  },
  "t_min": -2.0,
  "t_max": 8.0,
  "t_steps": 200,
  "z_min": 0.0,
  "z_max": 1.0,
  "z_steps": 10,
  "z_steps_inner": 2,
  "interaction_strengths": [2.0, 0.0],
  "velocity_classes": {
    "thermal_width": 10.0,
    "thermal_delta_min": -30.0,
    "thermal_delta_max":  30.0,
    "thermal_delta_steps": 12,
    "thermal_delta_inner_min": -3.0,
    "thermal_delta_inner_max":  3.0,
    "thermal_delta_inner_steps": 6
  },
  "savefile": "mbs-lambda-cpt-coupling"
}
"""

mbs_c = mb_solve.MBSolve().from_json_str(mb_solve_json_coupling)
mbs_c.mbsolve(recalc=False)
print("Done")

## Absorption spectra: the CPT dip

The two traces are overlaid below. The broad Doppler-broadened background
is the same in both cases. With the coupling field on, a narrow
transparency window — the dark resonance — appears at $\delta_p = 0$.

In [ ]:
fig = plot.spectrum_overlay(
    [mbs_no_c, mbs_c],
    field_idx=0,
    labels=["coupling off", "coupling on (Ω_c = 5γ)"],
    freq_range=8,
)
fig.update_layout(title="CPT dark resonance in a Λ system: probe absorption")
fig.show(renderer='notebook_connected')

## Probe propagation with coupling

The space-time plot of the probe shows that the medium becomes
transparent for the resonant frequency component: the probe pulse
propagates with much less attenuation than in the bare two-level case.

In [ ]:
fig = plot.field_spacetime(mbs_c, field_idx=0)
fig.update_layout(title="Probe |Ω_p(z, t)| — CPT (Ω_c = 5γ)")
fig.show(renderer='notebook_connected')

## Dark-state population

Once the atoms are in the dark state they accumulate in the two ground
states. The plot below shows $\rho_{00}$ and $\rho_{22}$ at the input
face, illustrating the coherent redistribution of population driven by
the coupling field.

In [ ]:
fig = plot.population(mbs_c, state_indices=[0, 1, 2], z_idx=0)
fig.update_layout(title="Ground/excited populations at z = z_min (CPT, Ω_c = 5γ)")
fig.show(renderer='notebook_connected')

## Summary

- Without coupling: Doppler-broadened Lorentzian absorption of width
  $\sim 2\gamma \sqrt{1 + (\sigma_D/\gamma)^2}$.
- With coupling: a narrow CPT dip of width $\sim \gamma_g \ll \gamma$
  appears at the two-photon resonance. The dip depth approaches unity
  as the coupling strength increases.
- Atoms accumulate in the dark state: $\rho_{11} \to 0$, population is
  distributed between $|0\rangle$ and $|2\rangle$ with ratio set by
  $\Omega_p^2 : \Omega_c^2$.

## References

1. E. Arimondo and G. Orriols, *Non-absorbing atomic coherences by
   coherent two-photon transitions in a three-level optical pumping*,
   Lett. Nuovo Cimento **17**, 333 (1976). First observation of CPT.
2. G. Alzetta, A. Gozzini, L. Moi, G. Orriols, *An experimental method
   for the observation of r.f. transitions and laser beat resonances in
   oriented Na vapour*, Nuovo Cimento B **36**, 5 (1976).
3. E. Arimondo, *Coherent Population Trapping in Laser Spectroscopy*,
   Prog. Opt. **35**, 257 (1996). Comprehensive review.